<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:44px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Statistical Power · External Validation · Inference Only</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Dış Doğrulama Örnekleminin Büyütülmesi</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">Ablasyonda gözlenen +0,010'luk AUC farkının gerçek mi yoksa örneklem gürültüsü mü olduğunu ayırmak</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Eğitim:</b> yok — kayıtlı kontrol noktaları kullanılır</div>
    <div><b>Örneklem:</b> sınıf başına 400 → 1000 (üst küme)</div>
    <div><b>Süre:</b> ~15 dakika</div>
    <div><b>Kollar:</b> A / B / C, üçü birden</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Gerekçe:</b> Ablasyonda RSNA'da C − A = +0,0102 (SE 0,0077, p = 0,19) bulundu. Bu etki gerçekse, p &lt; 0,05'e ulaşmak için gereken örneklem ≈ 1.770 görüntüdür. Mevcut 800 yetersizdi. Bu notebook örneklemi büyütüp aynı soruyu yeterli güçle sorar. Yeni örneklem eskisinin <b>üst kümesidir</b> (aynı tohum, genişletilmiş dilim), dolayısıyla ilk 400 görüntü birebir aynıdır ve önceki sonuçlar iç tutarlılık denetimi olarak yeniden üretilir.
  </div>
</div>

## Kurulum

**Eklenecek girdiler**

| # | Sekme | Kimlik | Not |
|---|---|---|---|
| 1 | Datasets | *(kendi yüklediğin)* ablasyon kontrol noktaları | `ablation_vit_raw.pth`, `ablation_vit_roi.pth`, `ablation_vit_lung.pth` |
| 2 | Competitions | `rsna-pneumonia-detection-challenge` | |
| 3 | Datasets | `nih-chest-xrays/data` | |

Kontrol noktalarını yüklemek için: ablasyon koşumunun **Output** sekmesinden üç `.pth`
dosyasını indirip *New Dataset* olarak yükleyin (ör. ad: `ablation-vit-checkpoints`).
Eğitim veri kümesine **gerek yoktur** — bu notebook hiçbir şey eğitmez.

**Ayarlar:** GPU açık, Internet açık (segmentasyon modeli indirilir).

### Neden bu işe yarar

Örneklem büyütmek AUC'nin *beklenen değerini* değiştirmez, yalnızca **standart hatasını**
küçültür. İki olasılık vardır ve ikisi de bilgi verir:

- Fark korunur ve güven aralığı 0'ı dışlar → segmentasyonun küçük ama **gerçek** bir
  ayrıştırma katkısı var.
- Fark küçülür / aralık 0'ı içermeye devam eder → gözlenen +0,010 gürültüydü;
  "ayrıştırmada fark yok" sonucu **sağlamlaşır** (dar aralık, güçlü negatif bulgu).

Her iki durumda da makalede savunulabilir bir cümle kurulur. Şu anki durumda ise
"bilmiyoruz" demek zorundayız.

In [ ]:
import os, sys, gc, io, json, glob, time, math, random, zipfile, hashlib, warnings, types
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from collections import OrderedDict, Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

from sklearn.metrics import (roc_curve, auc, average_precision_score, f1_score,
                             confusion_matrix, brier_score_loss)
from scipy import stats as sp_stats

ARM_LABEL = OrderedDict([
    ("raw",  "A · Segmentasyonsuz"),
    ("roi",  "B · Yalnizca RoI kirpma"),
    ("lung", "C · Maske + RoI (onerilen)"),
])
ARM_COLOR = {"raw": "#B04A1E", "roi": "#C2900A", "lung": "#0D8FA2"}

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8A9499", "axes.labelcolor": "#1B2327", "text.color": "#1B2327",
    "xtick.color": "#5A686F", "ytick.color": "#5A686F",
    "grid.color": "#D8DFE1", "grid.linewidth": 0.7, "legend.frameon": False,
})
WORK = "/kaggle/working"


# ----------------------------- AYARLAR -----------------------------
SEED          = 42       # ornek secimi icin - ablasyonla AYNI kalmali
K_RSNA        = 1000     # RSNA sinif basina goruntu (onceki: 400)
K_NIH         = None     # None -> havuzun izin verdigi en buyuk deger (~1400)
N_BOOT        = 2000
ARMS          = ["raw", "roi", "lung"]
NESTED_STEPS  = [200, 400, 600, 800, 1000]   # ic ice orneklem kararlilik egrisi
# --------------------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ablasyon kosumunda olculen degerler - iç tutarlilik denetimi icin
REF_400 = {"RSNA": {"raw": 0.916272, "roi": 0.918225, "lung": 0.926447},
           "NIH":  {"raw": 0.678675, "roi": 0.678550, "lung": 0.688656}}

print("=" * 62)
print(f"  Cihaz : {device} | Torch {torch.__version__}")
print(f"  RSNA sinif basina : {K_RSNA}   (onceki kosum: 400)")
print(f"  NIH  sinif basina : {'havuz kadar' if K_NIH is None else K_NIH}")
print("=" * 62)

In [ ]:
# ── Girdilerin kesfi: kontrol noktalari + dis setler ─────────────────────
INPUT = "/kaggle/input"

def find_dir_with(fname, root=INPUT):
    for r, _, files in os.walk(root):
        if fname in files:
            return r
    return None

# Kontrol noktalari (.pth). Kaggle bazen .zip olarak saklar - gerekirse acilir.
ckpts = {}
for p in glob.glob(os.path.join(INPUT, "**", "*.pth"), recursive=True):
    b = os.path.basename(p).lower()
    for arm in ARMS:
        if f"_{arm}." in b or b.endswith(f"{arm}.pth"):
            ckpts[arm] = p

if len(ckpts) < len(ARMS):
    for z in glob.glob(os.path.join(INPUT, "**", "*.zip"), recursive=True):
        try:
            with zipfile.ZipFile(z) as zf:
                for n in zf.namelist():
                    if n.endswith(".pth"):
                        dst = os.path.join("/kaggle/working/_ckpt", os.path.basename(n))
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        if not os.path.exists(dst):
                            with open(dst, "wb") as f:
                                f.write(zf.read(n))
                        for arm in ARMS:
                            if f"_{arm}." in os.path.basename(n).lower():
                                ckpts[arm] = dst
        except Exception:
            continue

RSNA_BASE = find_dir_with("stage_2_detailed_class_info.csv")
NIH_BASE  = find_dir_with("Data_Entry_2017.csv")

print("Bulunanlar")
for arm in ARMS:
    print(f"  ckpt {arm:<5}: {ckpts.get(arm)}")
print(f"  RSNA       : {RSNA_BASE}")
print(f"  NIH        : {NIH_BASE}")

missing = [a for a in ARMS if a not in ckpts]
assert not missing, (f"Kontrol noktasi bulunamadi: {missing}. "
                     "Ablasyon ciktisindaki .pth dosyalarini dataset olarak ekleyin.")
assert RSNA_BASE or NIH_BASE, "Hicbir dis set bulunamadi."

In [ ]:
# ── Modellerin yuklenmesi ────────────────────────────────────────────────
def build_vit_inference(num_classes, dropout):
    m = models.vit_b_16(weights=None)
    in_f = m.heads.head.in_features
    m.heads.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    return m

MODELS, CFG, CLASS_TO_IDX = {}, None, None
for arm in ARMS:
    ck = torch.load(ckpts[arm], map_location=device, weights_only=False)
    CFG = ck["config"]; CLASS_TO_IDX = ck["class_to_idx"]
    m = build_vit_inference(CFG["num_classes"], CFG.get("dropout", 0.1)).to(device)
    m.load_state_dict(ck["model_state_dict"], strict=True)
    MODELS[arm] = m.eval()
    print(f"  {arm:<5} yuklendi | kol etiketi='{ck.get('arm')}' | en iyi Val F1={ck.get('best_val_f1'):.4f}")

PNEU_IDX = CLASS_TO_IDX["PNEUMONIA"]
print(f"\nSiniflar: {CLASS_TO_IDX} | pozitif indeks = {PNEU_IDX}")
print(f"On-isleme: dilate={CFG['mask_dilate_frac']} roi_pad={CFG['roi_pad_frac']} "
      f"feather={CFG['mask_feather']} fill={CFG['fill_mode']} raw_mode={CFG.get('raw_mode')}")

eval_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(CFG["mean"], CFG["std"]),
])

In [ ]:
import transformers
from transformers import AutoModel

print("ianpan/chest-x-ray-basic yukleniyor... (transformers", transformers.__version__, ")")

def _load_seg():
    return AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                     trust_remote_code=True).to(device).eval()

_orig_finalize = getattr(transformers.modeling_utils.PreTrainedModel,
                         "_finalize_model_loading", None)
try:
    if _orig_finalize is not None:
        def _safe_finalize(model, *a, **k):
            if not hasattr(model, "all_tied_weights_keys"):
                model.all_tied_weights_keys = {}
            return _orig_finalize(model, *a, **k)
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _safe_finalize
    seg_model = _load_seg()
except Exception as e:
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise
finally:
    if _orig_finalize is not None:
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _orig_finalize

print("Segmentasyon modeli hazir.")

In [ ]:
def load_image_any(path, short_max):
    '''PNG/JPG/DICOM -> (rgb_u8, gray_u8); kisa kenari short_max'a indirir.'''
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        import pydicom
        dcm = pydicom.dcmread(path)
        arr = dcm.pixel_array.astype(np.float32)
        arr -= arr.min()
        if arr.max() > 0:
            arr /= arr.max()
        if getattr(dcm, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = 1.0 - arr
        pil = Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    else:
        pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), np.asarray(pil.convert("L"))


@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    '''Sag + sol akciger maskesi; kalp (sinif 3) dislanir.'''
    x = seg_model.preprocess(gray_u8)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    logits = seg_model(x)["mask"]
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()
    return ((pred == 1) | (pred == 2)).astype(np.uint8)


def _center_square(img, S, interp):
    H, W = img.shape[:2]
    s = 256.0 / min(H, W)
    rz = cv2.resize(img, (max(S, int(round(W * s))), max(S, int(round(H * s)))), interpolation=interp)
    h2, w2 = rz.shape[:2]
    y0, x0 = (h2 - S) // 2, (w2 - S) // 2
    return rz[y0:y0 + S, x0:x0 + S]


def prep_arms(rgb_u8, lung_u8, cfg):
    '''Tek segmentasyon cikisindan uc kolun girdisi. Ablasyon notebook'u ile birebir ayni.'''
    H, W = lung_u8.shape
    short = min(H, W)
    S = cfg["img_size"]
    out = {}

    if cfg.get("raw_mode", "resize") == "centercrop":
        a_img = _center_square(rgb_u8, S, cv2.INTER_AREA)
        a_msk = _center_square(lung_u8, S, cv2.INTER_NEAREST)
    else:
        a_img = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        a_msk = cv2.resize(lung_u8, (S, S), interpolation=cv2.INTER_NEAREST)
    out["raw"] = (a_img, a_msk.astype(np.uint8), True)

    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        ones = np.ones((S, S), np.uint8)
        out["roi"] = (fb, ones, False)
        out["lung"] = (fb, ones, False)
        return out

    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)

    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0:
            f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]

    fill = (np.array([m * 255.0 for m in cfg["mean"]], dtype=np.float32)
            if cfg["fill_mode"] == "mean" else np.zeros(3, dtype=np.float32))
    masked = (rgb_u8.astype(np.float32) * soft +
              fill[None, None, :] * (1.0 - soft)).clip(0, 255).astype(np.uint8)

    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)

    bh, bw = (y1 - y0 + 1), (x1 - x0 + 1)
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side);     tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side);     tx0 = max(0, tx1 - side)

    m224 = cv2.resize(mask_d[ty0:ty1, tx0:tx1], (S, S),
                      interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    out["roi"]  = (cv2.resize(rgb_u8[ty0:ty1, tx0:tx1], (S, S),
                              interpolation=cv2.INTER_AREA), m224, True)
    out["lung"] = (cv2.resize(masked[ty0:ty1, tx0:tx1], (S, S),
                              interpolation=cv2.INTER_AREA), m224, True)
    return out

print("On-isleme hatti hazir (ablasyon notebook'u ile birebir ayni).")

## Örneklem: eskisinin üst kümesi

Görüntüler önceki koşumdaki **aynı tohumla** karıştırılır ve dilim `[:400]` yerine
`[:1000]` alınır. Karıştırma deterministik olduğundan yeni örneklemin **ilk 400 görüntüsü
eskisiyle birebir aynıdır**. Bu iki şeyi birden sağlar:

1. Önceki sonuçlar (RSNA AUC 0,916 / 0,918 / 0,926) bu notebook içinde yeniden üretilerek
   hattın doğru kurulduğu doğrulanır.
2. Büyütme, örneklem değiştirmeden yalnızca **ekleme** yapar; farkın yönü örneklem
   seçiminden değil, gerçek etkiden gelir.

In [ ]:
def build_rsna_items(base, k, seed=SEED):
    info = pd.read_csv(os.path.join(base, "stage_2_detailed_class_info.csv")).drop_duplicates("patientId")
    img_dir = os.path.join(base, "stage_2_train_images")
    pos = info[info["class"] == "Lung Opacity"]["patientId"].tolist()
    neg = info[info["class"] == "Normal"]["patientId"].tolist()
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    kk = min(k, len(pos), len(neg))
    items  = [(os.path.join(img_dir, p + ".dcm"), PNEU_IDX)     for p in pos[:kk]]
    items += [(os.path.join(img_dir, p + ".dcm"), 1 - PNEU_IDX) for p in neg[:kk]]
    return items, kk, len(pos), len(neg)


def build_nih_items(base, k, seed=SEED):
    df = pd.read_csv(os.path.join(base, "Data_Entry_2017.csv"))
    labels = df["Finding Labels"].astype(str)
    is_pneu   = labels.apply(lambda s: "Pneumonia" in s.split("|"))
    is_normal = labels.apply(lambda s: s.strip() == "No Finding")
    index = {}
    for r, _, files in os.walk(base):
        for f in files:
            if f.lower().endswith(".png"):
                index[f] = os.path.join(r, f)
    def paths_for(mask_):
        return [index[n] for n in df[mask_]["Image Index"].tolist() if n in index]
    pos, neg = paths_for(is_pneu), paths_for(is_normal)
    rng = random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    kk = min(k if k else 10**9, len(pos), len(neg))
    items  = [(p, PNEU_IDX)     for p in pos[:kk]]
    items += [(p, 1 - PNEU_IDX) for p in neg[:kk]]
    return items, kk, len(pos), len(neg)


EXTERNAL = []
if RSNA_BASE:
    it, kk, np_, nn_ = build_rsna_items(RSNA_BASE, K_RSNA)
    EXTERNAL.append(("RSNA", it))
    print(f"RSNA : havuz {np_} pnomoni / {nn_} normal -> secilen {kk}/sinif = {len(it)} goruntu")
if NIH_BASE:
    it, kk, np_, nn_ = build_nih_items(NIH_BASE, K_NIH)
    EXTERNAL.append(("NIH", it))
    print(f"NIH  : havuz {np_} pnomoni / {nn_} normal -> secilen {kk}/sinif = {len(it)} goruntu")

# Ust kume dogrulamasi: ilk 400 eskisiyle ayni mi?
for name, items in EXTERNAL:
    pos = [p for p, l in items if l == PNEU_IDX]
    print(f"  {name}: ilk pozitif dosya = {os.path.basename(pos[0])}  "
          f"(onceki kosumla ayni olmali)")

In [ ]:
@torch.inference_mode()
def external_inference(name, items):
    for a in ARMS:
        MODELS[a].eval()
    y_true, probs, used = [], {a: [] for a in ARMS}, []
    t0 = time.time()
    for i, (path, label) in enumerate(items):
        try:
            rgb, gray = load_image_any(path, CFG["orig_short_max"])
            lung = lung_mask_ianpan(gray, gray.shape)
            arms = prep_arms(rgb, lung, CFG)
        except Exception:
            continue
        y_true.append(label); used.append(arms["lung"][2])
        for a in ARMS:
            x = eval_tf(Image.fromarray(arms[a][0])).unsqueeze(0).to(device)
            probs[a].append(torch.softmax(MODELS[a](x), 1)[0, PNEU_IDX].item())
        if (i + 1) % 500 == 0:
            print(f"    [{name}] {i+1}/{len(items)}  ({time.time()-t0:.0f} sn)")
    fb = 100.0 * (1.0 - np.mean(used)) if used else 0.0
    print(f"  [{name}] bitti: {len(y_true)} goruntu | {time.time()-t0:.0f} sn | geri cekilme %{fb:.1f}")
    yt = np.array(y_true)
    return {a: {"y_true": yt, "y_prob": np.array(probs[a])} for a in ARMS}


PRED = {}
for name, items in EXTERNAL:
    print(f"\n>>> {name} cikarimi ({len(items)} goruntu x 3 kol)...")
    PRED[name] = external_inference(name, items)

In [ ]:
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T + 1
    return T2


def _fast_delong(preds_sorted, m):
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    k = preds_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _midrank(pos[r, :])
        ty[r, :] = _midrank(neg[r, :])
        tz[r, :] = _midrank(preds_sorted[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    if k == 1:
        sx = np.array([[float(sx)]]); sy = np.array([[float(sy)]])
    return aucs, sx / m + sy / n


def delong_test(y_true, p1, p2):
    '''Iliskili iki ROC egrisi icin AUC farki testi -> (auc1, auc2, z, p).'''
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-y, kind="mergesort")
    m = int(y.sum())
    preds = np.vstack((np.asarray(p1), np.asarray(p2)))[:, order]
    aucs, cov = _fast_delong(preds, m)
    l = np.array([[1.0, -1.0]])
    var = float(l.dot(cov).dot(l.T))
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = float((aucs[0] - aucs[1]) / np.sqrt(var))
    return aucs[0], aucs[1], z, float(2 * (1 - sp_stats.norm.cdf(abs(z))))


def paired_bootstrap(y_true, prob_dict, n_boot=2000, seed=42):
    '''Sinif-katmanli yeniden ornekleme; ayni indeksler tum kollara uygulanir.'''
    rng = np.random.default_rng(seed)
    y = np.asarray(y_true)
    ip = np.where(y == 1)[0]; ineg = np.where(y == 0)[0]
    keys = list(prob_dict.keys())
    out = {a: np.empty(n_boot) for a in keys}
    for b in range(n_boot):
        ii = np.concatenate([rng.choice(ip, len(ip), replace=True),
                             rng.choice(ineg, len(ineg), replace=True)])
        yb = y[ii]
        for a in keys:
            fpr, tpr, _ = roc_curve(yb, np.asarray(prob_dict[a])[ii])
            out[a][b] = auc(fpr, tpr)
    return out


def mcnemar(y_true, p1, p2, thr=0.5):
    c1 = ((np.asarray(p1) >= thr).astype(int) == y_true)
    c2 = ((np.asarray(p2) >= thr).astype(int) == y_true)
    b = int(np.sum(c1 & ~c2)); c = int(np.sum(~c1 & c2))
    if b + c == 0:
        return b, c, 1.0
    return b, c, float(sp_stats.binomtest(b, b + c, 0.5).pvalue)


def expected_calibration_error(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); ece, N = 0.0, len(y_true)
    for i in range(n_bins):
        m = (y_prob > edges[i]) & (y_prob <= edges[i + 1])
        if m.sum() == 0:
            continue
        ece += (m.sum() / N) * abs(y_true[m].mean() - y_prob[m].mean())
    return ece


def metrics_block(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn + 1e-9); spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9); f1 = 2 * prec * sens / (prec + sens + 1e-9)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return {"N": len(y_true), "AUC": auc(fpr, tpr),
            "AP": average_precision_score(y_true, y_prob),
            "Acc": float((y_pred == y_true).mean()), "F1": f1,
            "Duyarlilik": sens, "Ozgulluk": spec,
            "Brier": brier_score_loss(y_true, y_prob),
            "ECE": expected_calibration_error(y_true, y_prob)}

print("Istatistik araclari hazir (DeLong yerel olarak dogrulandi: "
      "SE = bootstrap SD, oran 1.000).")

## İç tutarlılık denetimi

Yeni örneklemin ilk 400/sınıf alt kümesi, önceki koşumdaki örneklemle özdeştir.
Dolayısıyla o alt kümede hesaplanan AUC değerleri, ablasyon notebook'unun raporladığı
değerlerle **eşleşmelidir**. Eşleşmiyorsa ön-işleme veya model yükleme hattında bir
sapma var demektir ve devam edilmemelidir.

In [ ]:
def nested_subset(y_true, y_prob, n_per_class):
    '''Ilk n pozitif ve ilk n negatifi (item sirasina gore) secer.'''
    y = np.asarray(y_true)
    ip = np.where(y == 1)[0][:n_per_class]
    ineg = np.where(y == 0)[0][:n_per_class]
    ii = np.concatenate([ip, ineg])
    return y[ii], np.asarray(y_prob)[ii]


print("=" * 78)
print("  IC TUTARLILIK DENETIMI  (ilk 400/sinif - onceki kosumla ayni goruntuler)")
print("=" * 78)
print(f"{'kume':<8}{'kol':<7}{'bu kosum':>11}{'onceki':>10}{'fark':>10}")
ok = True
for name in PRED:
    for a in ARMS:
        yt, yp = nested_subset(PRED[name][a]["y_true"], PRED[name][a]["y_prob"], 400)
        fpr, tpr, _ = roc_curve(yt, yp); v = auc(fpr, tpr)
        ref = REF_400.get(name, {}).get(a, np.nan)
        d = v - ref
        flag = "" if abs(d) < 2e-3 else "  <-- SAPMA"
        if abs(d) >= 2e-3:
            ok = False
        print(f"{name:<8}{a:<7}{v:>11.4f}{ref:>10.4f}{d:>+10.4f}{flag}")
print("=" * 78)
print("  Tum farklar |0.002| altindaysa hat dogru kurulmustur." if ok else
      "  UYARI: sapma var - on-isleme veya kontrol noktasi uyusmuyor olabilir.")

In [ ]:
# ── Buyutulmus orneklemde tam metrikler ──────────────────────────────────
DS_LABEL = {"RSNA": "RSNA (yetiskin)", "NIH": "NIH ChestX-ray14"}
rows = []
for name in PRED:
    for a in ARMS:
        d = PRED[name][a]
        rows.append({"kume": DS_LABEL[name], "kol": a, **metrics_block(d["y_true"], d["y_prob"])})
df_all = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print("=" * 100)
print("  BUYUTULMUS ORNEKLEM - TUM METRIKLER  (esik 0,50)")
print("=" * 100)
print(df_all.to_string(index=False))
print("=" * 100)

# ── Bootstrap + eslestirilmis testler ────────────────────────────────────
PAIRS = [("lung", "raw"), ("lung", "roi"), ("roi", "raw")]
CI, BOOT, srows = {}, {}, []
for name in PRED:
    d = PRED[name]
    yt = d[ARMS[0]]["y_true"]
    BOOT[name] = paired_bootstrap(yt, {a: d[a]["y_prob"] for a in ARMS}, n_boot=N_BOOT, seed=SEED)
    CI[name] = {a: (float(np.percentile(BOOT[name][a], 2.5)),
                    float(np.percentile(BOOT[name][a], 97.5))) for a in ARMS}
    for a, b in PAIRS:
        A1, A2, z, p = delong_test(yt, d[a]["y_prob"], d[b]["y_prob"])
        diff = BOOT[name][a] - BOOT[name][b]
        nb, nc, pmc = mcnemar(yt, d[a]["y_prob"], d[b]["y_prob"])
        srows.append({"kume": DS_LABEL[name], "karsilastirma": f"{a} - {b}",
                      "N": len(yt), "dAUC": A1 - A2,
                      "GA alt": float(np.percentile(diff, 2.5)),
                      "GA ust": float(np.percentile(diff, 97.5)),
                      "DeLong z": z, "DeLong p": p,
                      "McNemar b/c": f"{nb}/{nc}", "McNemar p": pmc,
                      "anlamli": "EVET" if (p < 0.05) else "hayir"})
    print(f"  {DS_LABEL[name]}: bootstrap tamam (n={len(yt)})")

df_stat = pd.DataFrame(srows)
print("\n" + "=" * 118)
print("  ESLESTIRILMIS KARSILASTIRMALAR - BUYUTULMUS ORNEKLEM")
print("=" * 118)
print(df_stat.to_string(index=False))
print("=" * 118)

In [ ]:
# ── Figur: ic ice orneklem kararlilik egrisi ─────────────────────────────
# Her n icin AUC yeniden hesaplanir; orneklem ic ice oldugundan egri
# "daha fazla veri eklendikce tahmin nereye yakinsiyor" sorusunu yanitlar.
fig, axes = plt.subplots(1, len(PRED), figsize=(6.0 * len(PRED), 4.2), squeeze=False)
for ax, name in zip(axes[0], PRED):
    nmax = int((np.asarray(PRED[name][ARMS[0]]["y_true"]) == 1).sum())
    steps = [n for n in NESTED_STEPS if n <= nmax] or [nmax]
    if steps[-1] != nmax:
        steps = steps + [nmax]
    for a in ARMS:
        vals, los, his = [], [], []
        for n in steps:
            yt, yp = nested_subset(PRED[name][a]["y_true"], PRED[name][a]["y_prob"], n)
            fpr, tpr, _ = roc_curve(yt, yp); vals.append(auc(fpr, tpr))
            bb = paired_bootstrap(yt, {"x": yp}, n_boot=400, seed=SEED)["x"]
            los.append(np.percentile(bb, 2.5)); his.append(np.percentile(bb, 97.5))
        ax.plot(steps, vals, "-o", ms=4.5, lw=2, color=ARM_COLOR[a], label=ARM_LABEL[a], zorder=3)
        ax.fill_between(steps, los, his, color=ARM_COLOR[a], alpha=0.12, lw=0, zorder=2)
    ax.set_title(f"{DS_LABEL[name]} — örneklem büyüdükçe AUC")
    ax.set_xlabel("sinif basina goruntu"); ax.set_ylabel("ROC-AUC")
    ax.grid(alpha=.45); ax.set_axisbelow(True)
    ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "power_fig_01_nested.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figur: etki buyuklugu, 400 vs buyutulmus ─────────────────────────────
fig, ax = plt.subplots(figsize=(9.4, 3.2 + 0.42 * len(df_stat)))
labels, items = [], []
for name in PRED:
    for a, b in PAIRS:
        r400_yt, _ = nested_subset(PRED[name][a]["y_true"], PRED[name][a]["y_prob"], 400)
        _, p400a = nested_subset(PRED[name][a]["y_true"], PRED[name][a]["y_prob"], 400)
        _, p400b = nested_subset(PRED[name][b]["y_true"], PRED[name][b]["y_prob"], 400)
        bb = paired_bootstrap(r400_yt, {"a": p400a, "b": p400b}, n_boot=600, seed=SEED)
        d400 = bb["a"] - bb["b"]
        r = df_stat[(df_stat.kume == DS_LABEL[name]) &
                    (df_stat.karsilastirma == f"{a} - {b}")].iloc[0]
        labels.append(f"{DS_LABEL[name]}\n{a} - {b}")
        items.append((float(np.mean(d400)), float(np.percentile(d400, 2.5)),
                      float(np.percentile(d400, 97.5)),
                      r["dAUC"], r["GA alt"], r["GA ust"], r["DeLong p"], a))

y = np.arange(len(labels))
for i, (d0, l0, h0, d1, l1, h1, p, a) in enumerate(items):
    ax.plot([l0, h0], [i - 0.16, i - 0.16], lw=2, color="#9AA6AB", solid_capstyle="round", zorder=3)
    ax.plot([d0], [i - 0.16], "o", ms=6, color="#9AA6AB", mec="white", mew=1.2, zorder=4)
    ax.plot([l1, h1], [i + 0.16, i + 0.16], lw=2.4, color=ARM_COLOR[a], solid_capstyle="round", zorder=3)
    ax.plot([d1], [i + 0.16], "o", ms=7, color=ARM_COLOR[a], mec="white", mew=1.4, zorder=4)
    star = "*" if (not np.isnan(p) and p < 0.05) else ""
    ax.text(max(h0, h1) + 0.004, i, f"{d1:+.3f}{star}", va="center", fontsize=8.4)
ax.axvline(0, color="#5A686F", lw=1.1, zorder=2)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("AUC farki (%95 eslestirilmis bootstrap GA)")
ax.xaxis.grid(True, alpha=.5); ax.set_axisbelow(True); ax.invert_yaxis()
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], color="#9AA6AB", lw=2, marker="o", label="n = 400/sinif (onceki)"),
                   Line2D([], [], color="#0D8FA2", lw=2.4, marker="o", label="buyutulmus orneklem")],
          loc="lower right", fontsize=8)
fig.suptitle("Orneklem buyutmenin etkisi: nokta tahmini ayni kalirken aralik daralir  (* p < 0,05)",
             fontsize=10.5, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "power_fig_02_effect.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Kayit ────────────────────────────────────────────────────────────────
df_all.to_csv(os.path.join(WORK, "power_metrics.csv"), index=False)
df_stat.to_csv(os.path.join(WORK, "power_pairwise_tests.csv"), index=False)
pd.DataFrame([{"kume": DS_LABEL[n], "kol": a, "AUC_GA_alt": CI[n][a][0], "AUC_GA_ust": CI[n][a][1]}
              for n in PRED for a in ARMS]).to_csv(os.path.join(WORK, "power_auc_ci.csv"), index=False)
np.savez_compressed(os.path.join(WORK, "power_predictions.npz"),
                    **{f"{n}__{a}__{k}": PRED[n][a][k]
                       for n in PRED for a in ARMS for k in ("y_true", "y_prob")})

sig = df_stat[(df_stat["DeLong p"] < 0.05)]
print("\n" + "=" * 78)
print("  SONUC")
print("=" * 78)
for n in PRED:
    r = df_stat[(df_stat.kume == DS_LABEL[n]) & (df_stat.karsilastirma == "lung - raw")].iloc[0]
    print(f"  {DS_LABEL[n]:<20} n={r['N']:<5} C-A = {r['dAUC']:+.4f}  "
          f"GA [{r['GA alt']:+.4f}, {r['GA ust']:+.4f}]  p={r['DeLong p']:.3f}  -> {r['anlamli']}")
print("-" * 78)
print(f"  Anlamli cikan karsilastirma sayisi: {len(sig)}/{len(df_stat)}")
print("=" * 78)
for f in sorted(os.listdir(WORK)):
    if f.startswith("power"):
        print(f"  {f:<30} {os.path.getsize(os.path.join(WORK, f))/1e6:>7.2f} MB")

## Sonucun okunması

**İç tutarlılık denetimi sapma gösteriyorsa** başka hiçbir sayıya bakmayın — ön-işleme
veya kontrol noktası uyuşmuyor demektir.

**C − A anlamlı çıktıysa (GA 0'ı dışlıyor):** segmentasyonun küçük ama gerçek bir
ayrıştırma katkısı vardır. Makalede etki büyüklüğü (≈ +0,01 AUC) ve güven aralığıyla
birlikte raporlanmalı; "büyük bir kazanç" olarak sunulmamalıdır.

**Anlamsız kaldıysa:** artık *dar* bir aralıkla "fark yok" denebilir. Bu, önceki
"bilmiyoruz" durumundan çok daha güçlü bir ifadedir — makalede
*"segmentasyon ayrıştırma başarımını değiştirmez (ΔAUC 95% GA: [x, y]), buna karşılık
akciğer-odak oranını 0,005'ten 0,909'a çıkarır"* şeklinde kurulabilir.

> **Not.** Bu notebook eğitim tohumunu değiştirmez; ölçtüğü belirsizlik yalnızca
> **örneklem** belirsizliğidir. Eğitim rastgeleliğinden gelen belirsizlik için çoklu
> tohum notebook'una bakınız — iki kaynak birbirinin yerine geçmez.